* 웹 크롤링

정적 페이지 ; HTML이 완전히 완성된 상태
동적 페이지 ; 실시간으로 정보가 바뀌는 상태


In [ ]:
# # 정적 페이지 크롤링
# %conda install requests beautifulsoup4 lxml

# # 동적 페이지 크롤링
# %conda install selenium

# chromeDriver 다운로드
# %conda install webdriver-manager

# # 데이터 처리
# # %conda install pandas

%conda install requests beautifulsoup4 lxml selenium pandas

In [ ]:
import requests
from bs4 import BeautifulSoup

# 1. http 요청 (error일경우 https에서 http로)   request ; html로 문자열 가져옴
url = 'http://example.com/'
response = requests.get(url)

response.text

# 2. html 파싱                              beautifulsoup은 객체 형태로 변환 
# 특정 데이터 안에서 원하는 것을 찾는 행위
soup = BeautifulSoup(response.text,'lxml')
type(soup)

# 3. 데이터 추출
type(soup.find('h1'))           # 태그
title = soup.find('h1').text            # h태그 안에 들어가있는 텍스트 추출
print(title)


Example Domain


In [2]:
from bs4 import BeautifulSoup

html = """
<html>
  <head><title>샘플 페이지</title></head>
  <body>
    <h1 class="main-title">제목</h1>
    <div id="content">
      <p class="text">첫 번째 문단</p>
      <p class="text">두 번째 문단</p>
      <a href="/page1">링크1</a>
      <a href="/page2">링크2</a>
    </div>
  </body>
</html>
"""
soup = BeautifulSoup(html, 'lxml')

In [ ]:
# 태그로 데이터 찾기 메서드    ;   <>로 묶여 있는게 태그


title = soup.find('h1')
print(title.text)

for p in soup.find_all('p'):
    print(p.text)

제목
첫 번째 문단
두 번째 문단


In [4]:
soup.find_all('p')          # find는 가장 먼저 나오는 1개, find_all는 전체

[<p class="text">첫 번째 문단</p>, <p class="text">두 번째 문단</p>]

In [ ]:
# css 선택자 (select)

# class로 찾기
tx = soup.select('.text')        # class가 text인 값  

# id로 찾기                       # id가 content인 값
cont = soup.select('#content')

[<div id="content">
 <p class="text">첫 번째 문단</p>
 <p class="text">두 번째 문단</p>
 <a href="/page1">링크1</a>
 <a href="/page2">링크2</a>
 </div>]

In [ ]:
# 복합 선택자
soup.select('div#content a')    # div태그에 id=content 중에서 a태그

[<a href="/page1">링크1</a>, <a href="/page2">링크2</a>]

In [15]:
# 태그명으로 찾기
soup.select('p')

[<p class="text">첫 번째 문단</p>, <p class="text">두 번째 문단</p>]

In [22]:
html = """
<html>
  <body>
    <a href="/page1">링크1</a>
    <a href="/page2">링크2</a>
    <img src="/image1.jpg" alt="이미지1">
    <img src="/image2.jpg" alt="이미지2">
  </body>
</html>
"""

# soup = BeautifulSoup(html,'lxml')
soup = BeautifulSoup(html, 'html.parser')

# href 속성 가져오기
link = soup.find('a')
link['href']
print(link('href'))
print(link.get('href'))     #안전

# 모든 링크 추출
for p in soup.select('a'):
    print(p.get('href'))

[]
/page1
/page1
/page2


In [ ]:
# 실제 사이트 가져오기 -  https://books.toscrape.com/

import requests
from bs4 import BeautifulSoup
url = 'https://books.toscrape.com/'
response = requests.get(url)
response.raise_for_status()  # 에러체크

soup = BeautifulSoup(response.text, 'lxml')

# soup.select('article.product_pod')[0].find('h3').get('title')
soup.select('article.product_pod')[0].select_one('h3 a').get('title')



'A Light in the Attic'

In [44]:
import requests
from bs4 import BeautifulSoup
url = 'https://books.toscrape.com/'
response = requests.get(url)
response.raise_for_status()  # 에러체크

soup = BeautifulSoup(response.text, 'lxml')
titles, links = [],[]

# soup.select('article.product_pod')[0].find('h3').get('title')
for li in soup.select('ol.row li'):
    # print(li.select('article.product_pod')[0].select_one('h3 a').get('title'))
    titles.append(li.select('article.product_pod')[0].select_one('h3 a').get('title'))
    links.append('https://books.toscrape.com/'+ li.select('article.product_pod')[0].select_one('h3 a').get('href'))

In [45]:
data = {
    'title' : titles,
    'links' : links
}

pd.DataFrame(data)

,title,links
0,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,https://books.toscrape.com/catalogue/the-black...


In [51]:
# 모든 페이지 다 가져오기

import requests
from bs4 import BeautifulSoup
titles, links = [], []

for pagenum in range(1,51):
    url = f'https://books.toscrape.com/catalogue/page-{pagenum}.html'
    response = requests.get(url)
    response.raise_for_status()  # 에러체크

    soup = BeautifulSoup(response.text, 'lxml')

    for li in soup.select('ol.row li '):
        # print(li.select('article.product_pod')[0].select_one('h3 a').get('title'))
        titles.append(li.select('article.product_pod')[0].select_one('h3 a').get('title'))
        links.append( 'https://books.toscrape.com/'+ li.select('article.product_pod')[0].select_one('h3 a').get('href') )

In [69]:
# recordTable brand

import requests
from bs4 import BeautifulSoup
url = 'https://news.naver.com/section/100'

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')
#newsct > div.section_component.as_section_headline._PERSIST_CONTENT > div.section_article.as_headline._TEMPLATE > div

result = soup.select_one('div.section_article.as_headline._TEMPLATE')
for data in result.select('ul>li'):
    print(data.select_one('strong.sa_text_strong').text)

후임자로 '김용' 지목 양문석…국힘 "아바타 공천 시도 중단하라" 비판
李대통령 "전기료 이대로면 손실 엄청나게 늘어…전기 절약 당부"
"한국은 손님"이라던 이란대사 "해협 통과시켜 주나" 묻자
대구 컷오프發 위기 경고음에 어수선한 국힘…일각 "원점 재검토"
장동혁, '강경 당권파' 박민영 미디어대변인 재임명…당내 우려에도 재임명
하나증권 "한국항공우주, KF-21 양산 본격화…목표가 ↑"
김남국 “유시민, 갈라치기 말라며 분열…소강 국면에 휘발유 부어”
정부 "추경안 31일 국회제출…고유가 국민부담 덜어드릴 것"
[영상] 안창호함, K잠수함 최초 태평양 횡단…60조 캐나다 수주전 출항
李대통령 지지율 69%로 취임 후 최고치…민주 46%·국힘 18%[NBS](종합)
